# TrialMatch-Qwen — full reproducible run (Kaggle / Colab, free T4)
**Kaggle settings:** Accelerator = *GPU T4 x2* (or P100), Internet = *On* (needs a phone-verified account).
Total runtime ≈ 60–90 min. Stages are resumable: re-running skips finished work.

1. Put the repository in `/kaggle/working/repo` (Option A: git clone; Option B: upload the zip as a Kaggle *Dataset*).
2. Run all cells.  3. Download `results_bundle.zip` (and `adapters.zip`) from the Output panel.

In [ ]:
!nvidia-smi
import os; os.makedirs('/kaggle/working', exist_ok=True)

### Get the code (choose ONE option)

In [ ]:
# Option A: from your GitHub repository
REPO_URL = 'https://github.com/<your-user>/trialmatch-qwen.git'  # <-- edit
import os, subprocess
if not os.path.exists('/kaggle/working/repo') and '<your-user>' not in REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/repo'], check=True)

In [ ]:
# Option B: from an uploaded Kaggle dataset containing the zip or the folder
import glob, shutil, zipfile, os
if not os.path.exists('/kaggle/working/repo'):
    zips = glob.glob('/kaggle/input/**/trialmatch-qwen*.zip', recursive=True)
    dirs = glob.glob('/kaggle/input/**/trialmatch/__init__.py', recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall('/kaggle/working/')
        src = [d for d in glob.glob('/kaggle/working/*') if os.path.exists(d + '/trialmatch/__init__.py')][0]
        if src != '/kaggle/working/repo': shutil.move(src, '/kaggle/working/repo')
    elif dirs:
        shutil.copytree(os.path.dirname(os.path.dirname(dirs[0])), '/kaggle/working/repo')
print(os.listdir('/kaggle/working/repo'))

In [ ]:
%cd /kaggle/working/repo
!pip install -q -r requirements-kaggle.txt
!python -c "import transformers, peft, torch; print(transformers.__version__, peft.__version__, torch.__version__)"

### Run everything (split → sweep → fine-tune → before/after eval → attention → KV cache → speculative → figures)

In [ ]:
!bash scripts/run_all.sh 2>&1 | tee results/run_all.log

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3
!cat results/summary.json | head -c 3000

### Package outputs for download

In [ ]:
!cd /kaggle/working/repo && zip -qr /kaggle/working/results_bundle.zip results && zip -qr /kaggle/working/adapters.zip adapters_export
!ls -lh /kaggle/working/*.zip

### (Optional) Upload adapters to the Hugging Face Hub
Add a Kaggle secret named `HF_TOKEN` (write token) via *Add-ons → Secrets*, set your username, run:

In [ ]:
HF_USER = '<your-hf-username>'  # <-- edit
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no Kaggle secret:', e)
if '<' not in HF_USER:
    !python scripts/push_to_hub.py --user {HF_USER}

### (Optional) Launch the browser app with a public link (for the demo video)
Open the printed `https://xxxx.gradio.live` URL. Stop the cell to shut it down.

In [ ]:
!python -m app.app --share